# Module 02: Taxi/Ground-Delay Time Prediction (XGBoost)

Predicts how much longer an aircraft spends taxiing/holding as a function of runway queue depth, time of day, and wind. `core/twin_sim.py` calls this every time an aircraft reaches Taxiway Alpha, so a busy VABO queue visibly slows every aircraft down.

In [ ]:
!pip install -q xgboost==2.1.3 scikit-learn==1.5.2 pandas==2.2.3 numpy==1.26.4 joblib==1.4.2 requests==2.32.3


## 1. Fetch data and engineer a ground-delay-minutes target

Exact taxi-out timestamps require an authenticated BTS "on-time performance"
extract, so as a public, no-login proxy we use the same BTS-derived
carrier x airport x month delay-cause mirror and treat **NAS delay minutes
per flight** (`nas_delay / arr_flights`) as a stand-in for ground/taxi
congestion delay - it is, structurally, the same signal the digital twin
needs: "how much longer does an aircraft sit in the system because of
airport-side congestion." If the mirror's schema doesn't match, we fall back
to a physically-motivated synthetic dataset instead.

In [ ]:
import io
import numpy as np
import pandas as pd
import requests

DATA_URL = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Airline%20Delay.csv"

try:
    resp = requests.get(DATA_URL, timeout=20)
    resp.raise_for_status()
    raw = pd.read_csv(io.StringIO(resp.text))
    raw.columns = [c.strip().lower() for c in raw.columns]

    nas_col = next((c for c in raw.columns if "nas" in c and "ct" not in c), None)
    flights_col = next((c for c in raw.columns if "arr_flights" in c or c == "flights"), None)
    month_col = next((c for c in raw.columns if c == "month"), None)

    if not all([nas_col, flights_col, month_col]):
        raise ValueError("Expected columns not found in mirror - using synthetic dataset.")

    df = raw[[nas_col, flights_col, month_col]].copy()
    df.columns = ["nas_delay", "arr_flights", "month"]
    df = df.dropna()
    df = df[df["arr_flights"] > 0]
    df["ground_delay_min"] = (df["nas_delay"] / df["arr_flights"]).clip(0, 90)
    df["queue_depth"] = pd.qcut(df["arr_flights"], 10, labels=False, duplicates="drop")
    df["hour_of_day"] = np.random.default_rng(42).integers(0, 24, len(df))  # not in monthly aggregate
    df["wind_kt"] = np.random.default_rng(7).normal(10, 5, len(df)).clip(0, 40)

    X = df[["queue_depth", "hour_of_day", "wind_kt"]]
    y = df["ground_delay_min"]
    data_source = "YBI-Foundation/Dataset Airline Delay.csv (NAS-delay-per-flight proxy, live download)"

except Exception as exc:
    print(f"[fallback] Live dataset unavailable ({exc}). Generating a physically-motivated synthetic dataset.")
    rng = np.random.default_rng(42)
    n = 4000
    queue_depth = rng.integers(0, 12, n)
    hour_of_day = rng.integers(0, 24, n)
    wind_kt = rng.normal(10, 6, n).clip(0, 45)
    rush_hour_penalty = np.where(np.isin(hour_of_day, [7, 8, 9, 18, 19, 20]), 4.0, 0.0)
    y = 6 + queue_depth * 1.8 + rush_hour_penalty + wind_kt * 0.15 + rng.normal(0, 2.0, n)
    X = pd.DataFrame({"queue_depth": queue_depth, "hour_of_day": hour_of_day, "wind_kt": wind_kt})
    y = pd.Series(np.clip(y, 2, 90), name="ground_delay_min")
    data_source = "synthetic (rush-hour + queue-depth + wind physical model)"

print(f"Training rows: {len(X)} | source: {data_source}")
X.head()


## 2. Train + evaluate the XGBoost regressor

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
)
model.fit(X_train, y_train)

mae = mean_absolute_error(y_test, model.predict(X_test))
print(f"Test MAE: {mae:.2f} minutes of predicted ground delay")


## 3. Export for the live twin

In [ ]:
import joblib
from datetime import datetime, timezone

PKL_NAME = "02_taxi_time.pkl"
joblib.dump(
    {
        "model": model,
        "features": ["queue_depth", "hour_of_day", "wind_kt"],
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "data_source": data_source,
        "module": "02_taxi_time",
        "test_mae_minutes": float(mae),
    },
    PKL_NAME,
)
print(f"Saved {PKL_NAME}")


In [ ]:
# --- Download the trained artifact (Colab only; safe to run locally too) ---
try:
    from google.colab import files
    files.download(PKL_NAME)
    print(f"Downloading {PKL_NAME} ... move it into core/models/ on your machine.")
except ImportError:
    print(f"Not running in Colab - {PKL_NAME} is already saved in the current directory.")
    print("Copy it into core/models/ on your machine to activate this module in the live twin.")
